# 🎌 End-to-End Assignment: Anime Recommendations & Student Clustering
## Complete Machine Learning Pipeline — SVD Recommender + K-Means / Hierarchical / DBSCAN

> **Assignment Coverage**
> - **(a)** Eigenvalue & Eigenvector theory + role in PCA
> - **(b)** Student Social Network Profile Clustering (K-Means, Hierarchical, DBSCAN)
> - **(c)** Anime Recommendation System (Content-Based + Collaborative SVD)


---
## Part A — Eigenvalues, Eigenvectors & Their Role in PCA

### What is an Eigenvector?
Given a square matrix **A**, a non-zero vector **v** is an **eigenvector** if multiplying it by A only *scales* it — it does not rotate or shear it:

$$A \cdot v = \lambda \cdot v$$

The scalar **λ (lambda)** is the corresponding **eigenvalue** — it tells us *how much* the eigenvector is stretched or compressed.

### Intuitive Example
Imagine a transformation matrix that stretches the x-axis by 3 and the y-axis by 1.5.  
- The vector [1, 0] is an eigenvector with λ = 3 (it only gets scaled along x).  
- The vector [0, 1] is an eigenvector with λ = 1.5 (it only gets scaled along y).

### How PCA Uses Eigendecomposition
PCA finds directions of **maximum variance** in high-dimensional data.  
It computes the **covariance matrix** of the data, then finds its eigenvectors and eigenvalues:
- **Eigenvectors** become the *principal components* (new axes).
- **Eigenvalues** measure the *variance explained* by each component.
- Sorting by descending eigenvalue → top components capture the most information.
- Dropping low-eigenvalue components = dimensionality reduction with minimal information loss.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Numeric example of eigendecomposition ──────────────────────────────────
A = np.array([[4, 1],
              [2, 3]])

eigenvalues, eigenvectors = np.linalg.eig(A)
print("Matrix A:")
print(A)
print("\nEigenvalues:", eigenvalues)
print("Eigenvectors (columns):")
print(eigenvectors)

# Verification: A·v == λ·v
for i in range(len(eigenvalues)):
    lam = eigenvalues[i]
    v   = eigenvectors[:, i]
    print(f"\nλ={lam:.2f}  |  A·v = {A @ v}  |  λ·v = {lam * v}")
    print(f"  ✅ Match: {np.allclose(A @ v, lam * v)}")

# ── Visualise: original axes vs eigenvectors ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#FFF5F0')

colors = ['#E75480', '#6A5ACD']
origin = np.zeros(2)

for ax, title, vecs, labels in [
    (axes[0], "Standard Basis Vectors",
     [np.array([1, 0]), np.array([0, 1])], ["e₁=[1,0]", "e₂=[0,1]"]),
    (axes[1], "Eigenvectors of A",
     [eigenvectors[:, 0], eigenvectors[:, 1]],
     [f"v₁ (λ={eigenvalues[0]:.1f})", f"v₂ (λ={eigenvalues[1]:.1f})"])
]:
    for v, c, lbl in zip(vecs, colors, labels):
        ax.annotate("", xy=v, xytext=origin,
                    arrowprops=dict(arrowstyle="-|>", color=c, lw=2.5))
        ax.text(v[0]+0.05, v[1]+0.05, lbl, color=c, fontsize=11, fontweight='bold')
    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2)
    ax.axhline(0, color='grey', lw=0.8, ls='--')
    ax.axvline(0, color='grey', lw=0.8, ls='--')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    ax.set_facecolor('#FFF8F5')
    ax.grid(True, alpha=0.3)

plt.suptitle("Eigenvector Visualisation — Directions That Don't Rotate Under A",
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eigenvector_viz.png', dpi=120, bbox_inches='tight', facecolor='#FFF5F0')
plt.show()
print("\nFigure saved.")


: 

In [ ]:
# ── PCA demo: eigenvalues control how much variance each PC captures ─────
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

np.random.seed(42)
X_demo, _ = make_blobs(n_samples=300, centers=3, cluster_std=[1.5, 0.8, 1.2], random_state=42)
X_demo = StandardScaler().fit_transform(X_demo)

pca_demo = PCA()
pca_demo.fit(X_demo)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#FFF5F0')

# Explained variance bar
axes[0].bar(range(1, len(pca_demo.explained_variance_ratio_)+1),
            pca_demo.explained_variance_ratio_ * 100,
            color=['#E75480', '#6A5ACD'], edgecolor='white', linewidth=1.2)
axes[0].set_xlabel("Principal Component", fontsize=12)
axes[0].set_ylabel("Variance Explained (%)", fontsize=12)
axes[0].set_title("Variance Explained by Each PC\n(eigenvalue proportions)", fontsize=12, fontweight='bold')
axes[0].set_facecolor('#FFF8F5')

# Scree plot — cumulative
cum = np.cumsum(pca_demo.explained_variance_ratio_ * 100)
axes[1].plot(range(1, len(cum)+1), cum, 'o-', color='#E75480', lw=2.5, markersize=8)
axes[1].axhline(90, color='#6A5ACD', ls='--', lw=1.5, label='90% threshold')
axes[1].fill_between(range(1, len(cum)+1), cum, alpha=0.15, color='#E75480')
axes[1].set_xlabel("Number of Components", fontsize=12)
axes[1].set_ylabel("Cumulative Variance (%)", fontsize=12)
axes[1].set_title("Cumulative Variance (Scree Plot)\nChoose components above 90%", fontsize=12, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].set_facecolor('#FFF8F5')

for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle("PCA: Eigenvalues Determine How Much Each PC Matters",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('pca_eigenvalue_demo.png', dpi=120, bbox_inches='tight', facecolor='#FFF5F0')
plt.show()
print(f"Eigenvalues: {pca_demo.explained_variance_.round(3)}")
print(f"Explained ratios: {pca_demo.explained_variance_ratio_.round(3)}")


---
## Part B — Student Social Network Profile Clustering

### Dataset: Students' Social Network Profile Clustering
- **15,000 students** from a US high school social network (2006–2009)
- **40 features**: grad year, gender, age, friends count, plus word-count features  
  (sports, music, fashion, religion-related terms counted in their profiles)
- **Goal**: Segment students with similar interests & demographic profiles using  
  K-Means, Hierarchical, and DBSCAN — then compare Silhouette scores.


In [ ]:
# ── STEP 0: Imports ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings, re
warnings.filterwarnings('ignore')

from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.manifold import TSNE
from scipy.cluster.hierarchy import dendrogram, linkage
from wordcloud import WordCloud
from kneed import KneeLocator

# Pastel palette used throughout Part B
PASTEL = ['#FFB3C6', '#B5EAD7', '#C7CEEA', '#FFDAC1', '#FF9AA2',
          '#A8D8EA', '#E2F0CB', '#D4A5A5', '#957DAD', '#F3B0C3']
BG = '#FFF0F3'
print("✅ All libraries imported.")


In [ ]:
# ── STEP 1: Load Data ──────────────────────────────────────────────────────
df_raw = pd.read_csv('03_Clustering_Marketing.csv')
print(f"Shape  : {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
df_raw.head()


In [ ]:
# ── STEP 2a: Basic EDA ─────────────────────────────────────────────────────
print("=== Data Types & Nulls ===")
print(df_raw.info())
print("\n=== Missing Values ===")
print(df_raw.isnull().sum())
print("\n=== Descriptive Statistics ===")
df_raw.describe().T.round(2)


In [ ]:
# ── STEP 2b: EDA Visualisations ────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.patch.set_facecolor(BG)
fig.suptitle("Part B — Exploratory Data Analysis: Student Social Network Profiles",
             fontsize=15, fontweight='bold', y=1.01)

df_viz = df_raw.drop_duplicates().copy()

# (1) Gender distribution
if 'gender' in df_viz.columns:
    gd = df_viz['gender'].value_counts()
    axes[0,0].bar(gd.index, gd.values, color=['#FFB3C6','#B5EAD7','#C7CEEA'],
                  edgecolor='white', linewidth=1.5)
    axes[0,0].set_title("Gender Distribution", fontweight='bold')
    axes[0,0].set_facecolor('#FFF8FA')

# (2) Grad year distribution
df_viz['gradyear'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0,1], color=PASTEL[:4], edgecolor='white', linewidth=1.5)
axes[0,1].set_title("Students per Graduation Year", fontweight='bold')
axes[0,1].set_facecolor('#FFF8FA')
axes[0,1].tick_params(axis='x', rotation=0)

# (3) Age distribution
age_clean = pd.to_numeric(
    df_viz['age'].astype(str).apply(lambda x: re.sub(r'[A-Za-z]', '', x)),
    errors='coerce').dropna()
age_clean = age_clean[(age_clean >= 13) & (age_clean <= 22)]
axes[0,2].hist(age_clean, bins=20, color='#FFDAC1', edgecolor='white', linewidth=1.2)
axes[0,2].set_title("Age Distribution (13–22)", fontweight='bold')
axes[0,2].set_facecolor('#FFF8FA')
axes[0,2].set_xlabel("Age")

# (4) Friends count
axes[1,0].hist(df_viz['NumberOffriends'].clip(upper=200), bins=30,
               color='#C7CEEA', edgecolor='white', linewidth=1.2)
axes[1,0].set_title("Number of Friends (clipped @200)", fontweight='bold')
axes[1,0].set_facecolor('#FFF8FA')

# (5) Top word-count features — mean frequency
word_cols = [c for c in df_viz.columns if c not in ['gradyear','gender','age','NumberOffriends']]
word_means = df_viz[word_cols].mean().sort_values(ascending=False).head(15)
axes[1,1].barh(word_means.index[::-1], word_means.values[::-1],
               color=PASTEL[:15], edgecolor='white')
axes[1,1].set_title("Top 15 Word Features (Mean Frequency)", fontweight='bold')
axes[1,1].set_facecolor('#FFF8FA')

# (6) Correlation heatmap of top 12 numeric cols
top_cols = df_viz[word_cols].mean().nlargest(12).index.tolist()
corr = df_viz[top_cols].corr()
sns.heatmap(corr, ax=axes[1,2], cmap='RdPu', annot=False,
            linewidths=0.5, square=False, cbar_kws={'shrink':0.7})
axes[1,2].set_title("Correlation — Top 12 Word Features", fontweight='bold')

for ax in axes.flat:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_facecolor('#FFF8FA')

plt.tight_layout()
plt.savefig('clustering_eda.png', dpi=120, bbox_inches='tight', facecolor=BG)
plt.show()


In [ ]:
# ── STEP 2c: Word Cloud — Top Interest Keywords ────────────────────────────
word_cols = [c for c in df_raw.columns if c not in ['gradyear','gender','age','NumberOffriends']]
word_freq = df_raw[word_cols].sum().to_dict()

# Custom pastel colormap for word cloud
def pastel_color_func(word, font_size, position, orientation, random_state=None, **kwargs):
    colors = ['#E75480','#C9A0DC','#FF9AA2','#F4A460','#6A9FB5',
              '#85C1E9','#A9DFBF','#F1948A','#BB8FCE','#76D7C4']
    import random
    return random.choice(colors)

wc = WordCloud(
    width=1000, height=420,
    background_color='#FFF0F3',
    max_words=60,
    color_func=pastel_color_func,
    prefer_horizontal=0.85,
    collocations=False,
    font_path=None,
    min_font_size=10
).generate_from_frequencies(word_freq)

fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor('#FFF0F3')
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title("🌸 Student Profile Keywords — Word Cloud (size = total mentions)",
             fontsize=15, fontweight='bold', pad=15, color='#8B3A62')
plt.tight_layout()
plt.savefig('wordcloud_students.png', dpi=150, bbox_inches='tight', facecolor='#FFF0F3')
plt.show()


In [ ]:
# ── STEP 3: Clean, Encode & Impute ─────────────────────────────────────────
df = df_raw.drop_duplicates().copy()

# Fix age
df['age'] = pd.to_numeric(
    df['age'].astype(str).apply(lambda x: re.sub(r'[A-Za-z]', '', x)),
    errors='coerce')
df['age'] = df['age'].clip(lower=13, upper=22)

# Encode gender
df['gender'] = df['gender'].map({'M': 1, 'F': 0}).fillna(-1)

# Impute numerical columns with median
num_cols = df.select_dtypes(include='number').columns.tolist()
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

print(f"After cleaning — shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
df.head()


In [ ]:
# ── STEP 4: Skewness Check & log1p Transformation ─────────────────────────
skewness = df[num_cols].skew().sort_values(ascending=False)
print("=== Skewness (sorted) ===")
print(skewness.round(3))

high_skew = skewness[skewness.abs() > 1.0].index.tolist()
print(f"\nColumns with |skew| > 1.0 (applying log1p): {high_skew}")

df_t = df.copy()
for col in high_skew:
    if (df_t[col] >= 0).all():
        df_t[col] = np.log1p(df_t[col])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.patch.set_facecolor(BG)
skewness[:10].plot(kind='bar', ax=axes[0], color=PASTEL[:10], edgecolor='white')
axes[0].set_title("Top-10 Most Skewed Features (before transform)", fontweight='bold')
axes[0].axhline(1, color='red', ls='--', lw=1.2, label='skew=1')
axes[0].axhline(-1, color='red', ls='--', lw=1.2)
axes[0].legend(); axes[0].set_facecolor('#FFF8FA')

after_skew = df_t[num_cols].skew().sort_values(ascending=False)[:10]
after_skew.plot(kind='bar', ax=axes[1], color=PASTEL[:10], edgecolor='white')
axes[1].set_title("Skewness After log1p Transform", fontweight='bold')
axes[1].axhline(1, color='red', ls='--', lw=1.2)
axes[1].axhline(-1, color='red', ls='--', lw=1.2)
axes[1].set_facecolor('#FFF8FA')
plt.tight_layout()
plt.savefig('skewness_transform.png', dpi=120, bbox_inches='tight', facecolor=BG)
plt.show()


In [ ]:
# ── STEP 5: IQR Outlier Capping + RobustScaler ─────────────────────────────
df_clean = df_t.copy()
num_cols = df_clean.select_dtypes(include='number').columns.tolist()

capped = 0
for col in num_cols:
    Q1, Q3 = df_clean[col].quantile([0.25, 0.75])
    IQR    = Q3 - Q1
    lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_out  = ((df_clean[col] < lo) | (df_clean[col] > hi)).sum()
    if n_out > 0:
        df_clean[col] = df_clean[col].clip(lo, hi)
        capped += 1

print(f"Outlier capping applied to {capped} columns.")

scaler = RobustScaler()
X_scaled = scaler.fit_transform(df_clean[num_cols])
print(f"Scaled feature matrix shape: {X_scaled.shape}")


In [ ]:
# ── STEP 6: PCA — auto-select components for ≥ 90% variance ───────────────
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

cumvar    = np.cumsum(pca_full.explained_variance_ratio_)
n_comp_90 = int(np.argmax(cumvar >= 0.90)) + 1
print(f"Components needed for 90% variance: {n_comp_90}")

pca = PCA(n_components=n_comp_90, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA output shape: {X_pca.shape}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BG)

axes[0].bar(range(1, len(pca_full.explained_variance_ratio_)+1),
            pca_full.explained_variance_ratio_*100,
            color=PASTEL[0], edgecolor='white', alpha=0.85)
axes[0].axvline(n_comp_90, color='#8B3A62', ls='--', lw=2, label=f'90% threshold (PC={n_comp_90})')
axes[0].set_xlabel("Principal Component"); axes[0].set_ylabel("Variance (%)")
axes[0].set_title("Variance per PC (Scree Plot)", fontweight='bold')
axes[0].legend(); axes[0].set_facecolor('#FFF8FA')

axes[1].plot(range(1, len(cumvar)+1), cumvar*100, 'o-', color='#E75480', lw=2, ms=5)
axes[1].fill_between(range(1, len(cumvar)+1), cumvar*100, alpha=0.15, color='#E75480')
axes[1].axhline(90, color='#6A5ACD', ls='--', lw=1.8, label='90% variance')
axes[1].axvline(n_comp_90, color='#8B3A62', ls='--', lw=1.5)
axes[1].set_xlabel("Number of PCs"); axes[1].set_ylabel("Cumulative Variance (%)")
axes[1].set_title("Cumulative Explained Variance", fontweight='bold')
axes[1].legend(); axes[1].set_facecolor('#FFF8FA')

for ax in axes:
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('pca_variance.png', dpi=120, bbox_inches='tight', facecolor=BG)
plt.show()


In [ ]:
# ── STEP 7: K-Means — Elbow + Silhouette ──────────────────────────────────
inertia_list, sil_list = [], []
K_range = range(2, 12)

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=20, random_state=42)
    labels = km.fit_predict(X_pca)
    inertia_list.append(km.inertia_)
    sil_list.append(silhouette_score(X_pca, labels))

# KneeLocator for elbow
kl = KneeLocator(list(K_range), inertia_list, curve='convex', direction='decreasing')
best_k_elbow = kl.knee if kl.knee else 5
best_k = list(K_range)[np.argmax(sil_list)]   # Highest silhouette wins
print(f"Elbow K : {best_k_elbow}")
print(f"Best K (silhouette): {best_k}  →  silhouette = {max(sil_list):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BG)

axes[0].plot(K_range, inertia_list, 'o-', color='#E75480', lw=2, ms=7)
axes[0].axvline(best_k_elbow, color='#8B3A62', ls='--', lw=2, label=f'Elbow at K={best_k_elbow}')
axes[0].set_xlabel("Number of Clusters (K)"); axes[0].set_ylabel("Inertia (WCSS)")
axes[0].set_title("Elbow Method — Inertia vs K", fontweight='bold')
axes[0].legend(); axes[0].set_facecolor('#FFF8FA')

axes[1].plot(K_range, sil_list, 's-', color='#6A5ACD', lw=2, ms=7)
axes[1].axvline(best_k, color='#8B3A62', ls='--', lw=2, label=f'Best K={best_k}')
axes[1].set_xlabel("Number of Clusters (K)"); axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Score vs K", fontweight='bold')
axes[1].legend(); axes[1].set_facecolor('#FFF8FA')

for ax in axes:
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('kmeans_elbow_sil.png', dpi=120, bbox_inches='tight', facecolor=BG)
plt.show()


In [ ]:
# ── STEP 7b: Fit Final K-Means ─────────────────────────────────────────────
kmeans_model = KMeans(n_clusters=best_k, init='k-means++', n_init=20, random_state=42)
km_labels    = kmeans_model.fit_predict(X_pca)

km_sil = silhouette_score(X_pca, km_labels)
km_db  = davies_bouldin_score(X_pca, km_labels)
km_ch  = calinski_harabasz_score(X_pca, km_labels)

print(f"K-Means  |  K={best_k}  |  Silhouette={km_sil:.4f}  |  "
      f"Davies-Bouldin={km_db:.4f}  |  Calinski-Harabasz={km_ch:.1f}")


In [ ]:
# ── STEP 8: Hierarchical Clustering ────────────────────────────────────────
sil_hier = []
for k in range(2, 12):
    h = AgglomerativeClustering(n_clusters=k, linkage='ward')
    lbl = h.fit_predict(X_pca)
    sil_hier.append(silhouette_score(X_pca, lbl))

best_k_ag = list(range(2, 12))[np.argmax(sil_hier)]
print(f"Best K for Hierarchical: {best_k_ag}  →  silhouette = {max(sil_hier):.4f}")

# Dendrogram on a subsample (full 15k is slow)
sample_idx = np.random.choice(len(X_pca), 300, replace=False)
Z = linkage(X_pca[sample_idx], method='ward')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.patch.set_facecolor(BG)

# Silhouette curve
axes[0].plot(range(2, 12), sil_hier, 'd-', color='#E75480', lw=2, ms=7)
axes[0].axvline(best_k_ag, color='#8B3A62', ls='--', lw=2, label=f'Best K={best_k_ag}')
axes[0].set_xlabel("K"); axes[0].set_ylabel("Silhouette")
axes[0].set_title("Hierarchical — Silhouette vs K", fontweight='bold')
axes[0].legend(); axes[0].set_facecolor('#FFF8FA')

# Dendrogram
dendrogram(Z, ax=axes[1], truncate_mode='lastp', p=15,
           color_threshold=0, above_threshold_color='#C7CEEA',
           leaf_font_size=8)
axes[1].set_title("Dendrogram (300-sample, Ward linkage)", fontweight='bold')
axes[1].set_facecolor('#FFF8FA')

for ax in axes:
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('hierarchical_dendrogram.png', dpi=120, bbox_inches='tight', facecolor=BG)
plt.show()


In [ ]:
# ── STEP 8b: Fit Final Hierarchical ────────────────────────────────────────
hier_model  = AgglomerativeClustering(n_clusters=best_k_ag, linkage='ward')
hier_labels = hier_model.fit_predict(X_pca)

hier_sil = silhouette_score(X_pca, hier_labels)
hier_db  = davies_bouldin_score(X_pca, hier_labels)
hier_ch  = calinski_harabasz_score(X_pca, hier_labels)

print(f"Hierarchical  |  K={best_k_ag}  |  Silhouette={hier_sil:.4f}  |  "
      f"Davies-Bouldin={hier_db:.4f}  |  Calinski-Harabasz={hier_ch:.1f}")


In [ ]:
# ── STEP 9: DBSCAN — k-distance + grid search ──────────────────────────────
k_nn = 5
nn   = NearestNeighbors(n_neighbors=k_nn).fit(X_pca)
dist, _ = nn.kneighbors(X_pca)
k_dist  = np.sort(dist[:, k_nn-1])[::-1]

kl_db   = KneeLocator(range(len(k_dist)), k_dist, curve='convex', direction='decreasing')
knee_idx = kl_db.knee if kl_db.knee else len(k_dist)//10
eps_knee = round(k_dist[knee_idx], 3)
print(f"KneeLocator suggests eps ≈ {eps_knee}")

# Grid search
eps_candidates = np.linspace(max(0.1, eps_knee*0.5), eps_knee*2.0, 8).round(3)
min_samples_candidates = [3, 5, 7, 10]
best_sil_db, best_params, best_db_labels = -1, None, None

for eps in eps_candidates:
    for ms in min_samples_candidates:
        db = DBSCAN(eps=eps, min_samples=ms)
        lbl = db.fit_predict(X_pca)
        n_clust = len(set(lbl)) - (1 if -1 in lbl else 0)
        n_noise = (lbl == -1).sum()
        if n_clust >= 2 and n_noise < len(lbl)*0.5:
            mask = lbl != -1
            if mask.sum() > 50:
                s = silhouette_score(X_pca[mask], lbl[mask])
                if s > best_sil_db:
                    best_sil_db, best_params, best_db_labels = s, (eps, ms), lbl

print(f"Best DBSCAN params: eps={best_params[0]}, min_samples={best_params[1]}")
n_clusters_db = len(set(best_db_labels)) - (1 if -1 in best_db_labels else 0)
print(f"Clusters found: {n_clusters_db}  |  Noise points: {(best_db_labels==-1).sum()}")


In [ ]:
# ── STEP 9b: Evaluate Final DBSCAN ─────────────────────────────────────────
mask_db = best_db_labels != -1
dbscan_sil = silhouette_score(X_pca[mask_db], best_db_labels[mask_db])
dbscan_db  = davies_bouldin_score(X_pca[mask_db], best_db_labels[mask_db])
dbscan_ch  = calinski_harabasz_score(X_pca[mask_db], best_db_labels[mask_db])

print(f"DBSCAN  |  Clusters={n_clusters_db}  |  Silhouette={dbscan_sil:.4f}  |  "
      f"Davies-Bouldin={dbscan_db:.4f}  |  Calinski-Harabasz={dbscan_ch:.1f}")


In [ ]:
# ── STEP 10: Comparison Table ──────────────────────────────────────────────
results_df = pd.DataFrame({
    'Method'             : ['K-Means', 'Hierarchical', 'DBSCAN'],
    'Optimal Clusters'   : [best_k, best_k_ag, n_clusters_db],
    'Silhouette ↑'       : [round(km_sil,4), round(hier_sil,4), round(dbscan_sil,4)],
    'Davies-Bouldin ↓'   : [round(km_db,4),  round(hier_db,4),  round(dbscan_db,4)],
    'Calinski-Harabasz ↑': [round(km_ch,1),   round(hier_ch,1),  round(dbscan_ch,1)],
})
print(results_df.to_string(index=False))

# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor(BG)
x = np.arange(3)
bars = ax.bar(x, results_df['Silhouette ↑'], color=PASTEL[:3],
              edgecolor='white', linewidth=1.5, width=0.5)
ax.set_xticks(x)
ax.set_xticklabels(['K-Means', 'Hierarchical', 'DBSCAN'], fontsize=12)
ax.set_ylabel("Silhouette Score (higher = better)", fontsize=12)
ax.set_title("Clustering Methods — Silhouette Score Comparison", fontsize=13, fontweight='bold')
for bar, val in zip(bars, results_df['Silhouette ↑']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
            f"{val:.4f}", ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_facecolor('#FFF8FA')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('clustering_comparison.png', dpi=120, bbox_inches='tight', facecolor=BG)
plt.show()


In [ ]:
# ── STEP 11: t-SNE Visualisation of K-Means Clusters ──────────────────────
print("Running t-SNE (30–60 s on 15k points)...")
tsne     = TSNE(n_components=2, perplexity=40, random_state=42, n_iter=1000)
X_tsne   = tsne.fit_transform(X_pca)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor(BG)
fig.suptitle("t-SNE 2D Cluster Visualisations", fontsize=14, fontweight='bold')

label_sets = [
    (km_labels,   f"K-Means (K={best_k})",      'K-Means Cluster'),
    (hier_labels, f"Hierarchical (K={best_k_ag})", 'Hier. Cluster'),
    (best_db_labels, f"DBSCAN (clusters={n_clusters_db})", 'DBSCAN Cluster'),
]
for ax, (lbl, title, leg_title) in zip(axes, label_sets):
    unique = sorted(set(lbl))
    palette = plt.cm.get_cmap('Pastel1', len(unique))
    for i, cl in enumerate(unique):
        mask = lbl == cl
        label = f"Noise" if cl == -1 else f"Cluster {cl}"
        ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                   c=[palette(i)], s=5, alpha=0.7, label=label)
    ax.set_title(title, fontweight='bold')
    ax.set_facecolor('#FFF8FA')
    ax.tick_params(labelbottom=False, labelleft=False)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    if len(unique) <= 8:
        ax.legend(markerscale=4, fontsize=8, loc='best')

plt.tight_layout()
plt.savefig('tsne_clusters.png', dpi=120, bbox_inches='tight', facecolor=BG)
plt.show()


In [ ]:
# ── STEP 12: Cluster Profiles ──────────────────────────────────────────────
word_cols = [c for c in df.columns if c not in ['gradyear','gender','age','NumberOffriends']]
df_profile = df.copy()
df_profile['kmeans_cluster'] = km_labels

cluster_profile = df_profile.groupby('kmeans_cluster')[word_cols].mean()
print("=== K-Means Cluster Profiles (mean word-count features) ===")
print(cluster_profile.round(2))

# Heatmap
fig, ax = plt.subplots(figsize=(16, 5))
fig.patch.set_facecolor(BG)
sns.heatmap(cluster_profile, cmap='RdPu', annot=True, fmt='.2f',
            linewidths=0.4, ax=ax, cbar_kws={'label':'Mean Frequency'})
ax.set_title("Cluster Profiles — Mean Feature Values per K-Means Cluster",
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel("Word Features"); ax.set_ylabel("Cluster ID")
plt.tight_layout()
plt.savefig('cluster_profiles.png', dpi=120, bbox_inches='tight', facecolor=BG)
plt.show()


---
## Part C — Anime Recommendation System (SVD + Content-Based)

### Dataset: Anime Recommendations Database
- **12,294 anime** with genres, type, episodes, ratings, and member counts
- **Goal**: Build a content-based recommender using **TF-IDF on genres** and  
  a collaborative SVD recommender using the user-item rating matrix.
  
### Why SVD?
SVD (Singular Value Decomposition) factorises the user-item matrix into latent  
factor spaces, capturing hidden patterns like "action + adventure lovers"  
without explicit feature engineering. It is the backbone of Netflix-style recommendations.


In [ ]:
# ── STEP 0: Imports for Part C ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from wordcloud import WordCloud

PASTEL_C = ['#FFB3C6','#C7CEEA','#B5EAD7','#FFDAC1','#FF9AA2',
            '#A8D8EA','#E2F0CB','#D4A5A5','#957DAD','#F3B0C3']
BG_C = '#FFF5F0'
print("✅ Imports ready.")


In [ ]:
# ── STEP 1: Load Data ──────────────────────────────────────────────────────
anime = pd.read_csv('anime.csv')
print(f"Anime shape: {anime.shape}")
print(anime.head())


In [ ]:
# ── STEP 2: EDA ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.patch.set_facecolor(BG_C)
fig.suptitle("Part C — Anime Dataset EDA", fontsize=15, fontweight='bold', y=1.01)

# (1) Rating distribution
axes[0,0].hist(anime['rating'].dropna(), bins=30, color='#FFB3C6',
               edgecolor='white', linewidth=1.2)
axes[0,0].set_title("Rating Distribution", fontweight='bold')
axes[0,0].set_xlabel("Rating"); axes[0,0].set_facecolor('#FFF8FA')

# (2) Type distribution
type_counts = anime['type'].value_counts()
axes[0,1].bar(type_counts.index, type_counts.values, color=PASTEL_C[:len(type_counts)],
              edgecolor='white', linewidth=1.5)
axes[0,1].set_title("Anime Type Distribution", fontweight='bold')
axes[0,1].set_xlabel("Type"); axes[0,1].set_facecolor('#FFF8FA')
axes[0,1].tick_params(axis='x', rotation=15)

# (3) Members (log scale)
axes[0,2].hist(np.log1p(anime['members']), bins=30, color='#C7CEEA',
               edgecolor='white', linewidth=1.2)
axes[0,2].set_title("Member Count (log scale)", fontweight='bold')
axes[0,2].set_xlabel("log(1 + members)"); axes[0,2].set_facecolor('#FFF8FA')

# (4) Top 15 rated anime
top15 = anime.nlargest(15, 'rating')[['name','rating']].reset_index(drop=True)
axes[1,0].barh(top15['name'][::-1], top15['rating'][::-1],
               color=PASTEL_C[:15], edgecolor='white')
axes[1,0].set_title("Top 15 Highest Rated Anime", fontweight='bold')
axes[1,0].set_xlabel("Rating"); axes[1,0].set_facecolor('#FFF8FA')
axes[1,0].tick_params(axis='y', labelsize=8)

# (5) Top 15 most popular
top15m = anime.nlargest(15, 'members')[['name','members']].reset_index(drop=True)
axes[1,1].barh(top15m['name'][::-1], top15m['members'][::-1],
               color=PASTEL_C[:15], edgecolor='white')
axes[1,1].set_title("Top 15 Most Popular Anime (Members)", fontweight='bold')
axes[1,1].set_xlabel("Members"); axes[1,1].set_facecolor('#FFF8FA')
axes[1,1].tick_params(axis='y', labelsize=8)

# (6) Rating vs members scatter
sc = axes[1,2].scatter(np.log1p(anime['members']), anime['rating'],
                        c=anime['rating'], cmap='RdPu', s=8, alpha=0.6)
plt.colorbar(sc, ax=axes[1,2], label='Rating')
axes[1,2].set_title("Rating vs Popularity (log members)", fontweight='bold')
axes[1,2].set_xlabel("log(1 + members)"); axes[1,2].set_ylabel("Rating")
axes[1,2].set_facecolor('#FFF8FA')

for ax in axes.flat:
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('anime_eda.png', dpi=120, bbox_inches='tight', facecolor=BG_C)
plt.show()


In [ ]:
# ── STEP 2b: Genre Word Cloud ───────────────────────────────────────────────
from collections import Counter

all_genres = []
for genres in anime['genre'].dropna():
    all_genres.extend([g.strip() for g in genres.split(',')])

genre_freq = Counter(all_genres)
print("Top 20 Genres:")
for g, cnt in genre_freq.most_common(20):
    print(f"  {g}: {cnt}")

def pastel_anime_color(word, font_size, position, orientation, random_state=None, **kwargs):
    colors = ['#E75480','#BB8FCE','#FF9AA2','#F4A460','#6A9FB5',
              '#85C1E9','#F9A8D4','#F1948A','#C9A0DC','#76D7C4',
              '#FFB347','#77DD77','#AEC6CF']
    import random
    return random.choice(colors)

wc_anime = WordCloud(
    width=1100, height=430,
    background_color='#FFF5F0',
    max_words=80,
    color_func=pastel_anime_color,
    prefer_horizontal=0.85,
    collocations=False,
    min_font_size=9
).generate_from_frequencies(genre_freq)

fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor(BG_C)
ax.imshow(wc_anime, interpolation='bilinear')
ax.axis('off')
ax.set_title("🎌 Anime Genre Word Cloud (size = frequency across all anime)",
             fontsize=15, fontweight='bold', pad=15, color='#8B3A62')
plt.tight_layout()
plt.savefig('wordcloud_anime_genres.png', dpi=150, bbox_inches='tight', facecolor=BG_C)
plt.show()


In [ ]:
# ── STEP 3: Skewness Check & Transformation ────────────────────────────────
anime_num = anime[['rating','members','episodes']].copy()
anime_num['episodes'] = pd.to_numeric(anime_num['episodes'], errors='coerce')

skew_before = anime_num.skew()
print("Skewness before transform:")
print(skew_before.round(3))

anime_t = anime_num.copy()
for col in ['members','episodes']:
    anime_t[col] = np.log1p(anime_t[col].fillna(0))

skew_after = anime_t.skew()
print("\nSkewness after log1p:")
print(skew_after.round(3))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.patch.set_facecolor(BG_C)
fig.suptitle("Skewness & Transformation — Anime Numeric Features", fontsize=13, fontweight='bold')

for i, (col, title) in enumerate([
    ('rating',   'Rating'),
    ('members',  'Members'),
    ('episodes', 'Episodes')
]):
    # Before
    axes[0, i].hist(anime_num[col].dropna(), bins=30, color=PASTEL_C[i], edgecolor='white')
    axes[0, i].set_title(f"{title} — Before (skew={skew_before[col]:.2f})", fontweight='bold')
    axes[0, i].set_facecolor('#FFF8FA')
    # After
    axes[1, i].hist(anime_t[col].dropna(), bins=30, color=PASTEL_C[i+3], edgecolor='white')
    axes[1, i].set_title(f"{title} — After log1p (skew={skew_after[col]:.2f})", fontweight='bold')
    axes[1, i].set_facecolor('#FFF8FA')

for ax in axes.flat:
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('anime_skewness.png', dpi=120, bbox_inches='tight', facecolor=BG_C)
plt.show()


In [ ]:
# ── STEP 4: Content-Based Recommender (TF-IDF on Genres) ──────────────────
anime_cb = anime.dropna(subset=['genre']).copy().reset_index(drop=True)
anime_cb['genre_clean'] = anime_cb['genre'].str.replace(', ', ' ').str.replace('-', '')

tfidf = TfidfVectorizer(token_pattern=r"[a-zA-Z0-9]+", ngram_range=(1,1))
tfidf_matrix = tfidf.fit_transform(anime_cb['genre_clean'])
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

# Cosine similarity
cos_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
anime_idx = pd.Series(anime_cb.index, index=anime_cb['name'])

def recommend_content(title, n=10):
    """Return top-n similar anime based on genre TF-IDF cosine similarity."""
    if title not in anime_idx.index:
        return f"'{title}' not found in dataset."
    idx  = anime_idx[title]
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]
    sims = list(enumerate(cos_sim[idx]))
    sims = sorted(sims, key=lambda x: x[1], reverse=True)[1:n+1]
    result = anime_cb.iloc[[i[0] for i in sims]][['name','genre','rating','members']].copy()
    result['similarity'] = [round(s[1],4) for s in sims]
    return result.reset_index(drop=True)

# Demo
test_anime = 'Sword Art Online'
print(f"\nContent-Based Recommendations for: {test_anime}")
print(recommend_content(test_anime, 10))


In [ ]:
# ── STEP 5: Collaborative SVD Recommender (without rating.csv) ────────────
# We build a pseudo user-item matrix from the anime dataset itself
# using a synthetically generated small ratings table for demonstration,
# then apply TruncatedSVD for low-rank approximation.

np.random.seed(42)
# Use top-500 most popular anime for tractable demo
top_anime_names = anime.nlargest(500, 'members')['name'].values
n_users  = 300
n_items  = len(top_anime_names)

# Simulate sparse user-item rating matrix (0 = unrated)
user_item_sim = np.zeros((n_users, n_items))
for u in range(n_users):
    n_rated = np.random.randint(10, 80)
    items   = np.random.choice(n_items, n_rated, replace=False)
    # Ratings correlated with anime's actual mean rating
    base = anime.set_index('name').loc[top_anime_names, 'rating'].fillna(7).values
    user_item_sim[u, items] = np.clip(base[items] + np.random.randn(n_rated)*1.5, 1, 10)

print(f"User-Item matrix: {user_item_sim.shape}  |  "
      f"Density: {(user_item_sim>0).mean()*100:.1f}%")


In [ ]:
# ── STEP 6: SVD Fit & Evaluate ─────────────────────────────────────────────
# Compare RMSE for different numbers of latent factors
rmse_results = {}
for k in [5, 10, 20, 50, 100]:
    svd = TruncatedSVD(n_components=min(k, min(user_item_sim.shape)-1), random_state=42)
    U   = svd.fit_transform(user_item_sim)
    R   = np.dot(U, svd.components_)
    
    # Evaluate on observed entries only
    mask_obs = user_item_sim > 0
    actual   = user_item_sim[mask_obs]
    pred     = R[mask_obs]
    rmse     = np.sqrt(mean_squared_error(actual, pred))
    rmse_results[k] = round(rmse, 4)
    print(f"k={k:3d}  |  RMSE = {rmse:.4f}")

best_k_svd = min(rmse_results, key=rmse_results.get)
print(f"\nBest k = {best_k_svd}  (lowest RMSE = {rmse_results[best_k_svd]})")

# Final model with best k
svd_final  = TruncatedSVD(n_components=best_k_svd, random_state=42)
U_final    = svd_final.fit_transform(user_item_sim)
R_final    = np.dot(U_final, svd_final.components_)
rec_df     = pd.DataFrame(R_final, columns=top_anime_names)


In [ ]:
# ── STEP 7: SVD RMSE Plot + Explained Variance ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BG_C)

axes[0].plot(list(rmse_results.keys()), list(rmse_results.values()),
             'o-', color='#E75480', lw=2.5, ms=8)
axes[0].axvline(best_k_svd, color='#8B3A62', ls='--', lw=2,
                label=f'Best k={best_k_svd}')
axes[0].set_xlabel("Number of Latent Factors (k)", fontsize=12)
axes[0].set_ylabel("RMSE on Observed Entries", fontsize=12)
axes[0].set_title("SVD RMSE vs Number of Latent Factors", fontsize=12, fontweight='bold')
axes[0].legend(); axes[0].set_facecolor('#FFF8FA')
axes[0].grid(alpha=0.3)
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

# Explained variance ratio (singular values)
svd_ev = TruncatedSVD(n_components=50, random_state=42)
svd_ev.fit(user_item_sim)
cum_ev = np.cumsum(svd_ev.explained_variance_ratio_) * 100

axes[1].plot(range(1, 51), cum_ev, 'o-', color='#6A5ACD', lw=2, ms=5)
axes[1].fill_between(range(1, 51), cum_ev, alpha=0.15, color='#6A5ACD')
axes[1].axhline(80, color='#E75480', ls='--', lw=1.5, label='80% variance')
axes[1].axhline(90, color='#8B3A62', ls='--', lw=1.5, label='90% variance')
axes[1].set_xlabel("Number of SVD Components", fontsize=12)
axes[1].set_ylabel("Cumulative Explained Variance (%)", fontsize=12)
axes[1].set_title("SVD Cumulative Variance — How Many Factors We Need", fontsize=12, fontweight='bold')
axes[1].legend(); axes[1].set_facecolor('#FFF8FA')
axes[1].grid(alpha=0.3)
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('svd_rmse_variance.png', dpi=120, bbox_inches='tight', facecolor=BG_C)
plt.show()
print(f"RMSE summary: {rmse_results}")


In [ ]:
# ── STEP 8: SVD-based Collaborative Recommendations ───────────────────────
def recommend_svd(user_id, n=10):
    """Recommend top-n unseen anime for a given user via SVD predictions."""
    pred_ratings = rec_df.iloc[user_id]
    already_seen = top_anime_names[user_item_sim[user_id] > 0]
    candidates   = pred_ratings.drop(labels=already_seen, errors='ignore')
    top_n        = candidates.nlargest(n).reset_index()
    top_n.columns = ['Anime', 'Predicted Rating']
    top_n['Predicted Rating'] = top_n['Predicted Rating'].round(2)
    return top_n

print("SVD Recommendations for User 42:")
print(recommend_svd(42))


In [ ]:
# ── STEP 9: Anime Item-Item Similarity via SVD components ─────────────────
item_sim = cosine_similarity(svd_final.components_.T)
item_sim_df = pd.DataFrame(item_sim, index=top_anime_names, columns=top_anime_names)

def similar_anime_svd(title, n=10):
    """Find anime most similar to the given title using SVD latent space."""
    if title not in item_sim_df.index:
        return f"'{title}' not in top-500 anime."
    sims = item_sim_df[title].drop(title).nlargest(n)
    result = pd.DataFrame({'Anime': sims.index, 'Similarity': sims.values.round(4)})
    return result.reset_index(drop=True)

# Demo
test_title = 'Sword Art Online'
if test_title in item_sim_df.index:
    print(f"SVD Item-Item Similar to '{test_title}':")
    print(similar_anime_svd(test_title))
else:
    fallback = top_anime_names[10]
    print(f"Demo with '{fallback}':")
    print(similar_anime_svd(fallback))


In [ ]:
# ── STEP 10: Final Summary Visualisation — Top Genres & SVD Heatmap ────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor(BG_C)
fig.suptitle("Part C — Final Summary: Genre Landscape & SVD Latent Space",
             fontsize=14, fontweight='bold')

# (1) Top 20 genres bar chart
from collections import Counter
all_genres = []
for genres in anime['genre'].dropna():
    all_genres.extend([g.strip() for g in genres.split(',')])
genre_counts = pd.Series(Counter(all_genres)).sort_values(ascending=False)[:20]

axes[0].barh(genre_counts.index[::-1], genre_counts.values[::-1],
             color=PASTEL_C[:20], edgecolor='white', linewidth=1.2)
axes[0].set_title("Top 20 Anime Genres by Frequency", fontweight='bold')
axes[0].set_xlabel("Count"); axes[0].set_facecolor('#FFF8FA')
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)
axes[0].tick_params(axis='y', labelsize=9)

# (2) SVD component heatmap (first 10 components × top 20 anime)
top20_names = anime.nlargest(20, 'members')['name'].values
top20_idx   = [list(top_anime_names).index(n) for n in top20_names if n in top_anime_names]
heatmap_data = svd_final.components_[:8, top20_idx]

sns.heatmap(heatmap_data, ax=axes[1], cmap='RdPu',
            xticklabels=[n[:15]+'…' if len(n)>15 else n for n in top20_names[:len(top20_idx)]],
            yticklabels=[f"Factor {i+1}" for i in range(8)],
            linewidths=0.5, annot=False, cbar_kws={'label':'Factor Loading'})
axes[1].set_title("SVD Latent Factors × Top-20 Anime", fontweight='bold')
axes[1].tick_params(axis='x', rotation=45, labelsize=8)
axes[1].set_facecolor('#FFF8FA')

plt.tight_layout()
plt.savefig('anime_summary.png', dpi=120, bbox_inches='tight', facecolor=BG_C)
plt.show()


In [ ]:
# ── FINAL MODEL PERFORMANCE SUMMARY ────────────────────────────────────────
print("=" * 60)
print("         COMPLETE ASSIGNMENT — MODEL SUMMARY")
print("=" * 60)

print("\n📌 PART A — Eigenvalue/Eigenvector Demo")
print("   Verified: A·v = λ·v holds for all eigenpairs ✅")

print("\n📌 PART B — Student Clustering (15,000 students)")
print(f"   K-Means      | K={best_k:<2d} | Silhouette={km_sil:.4f}  | DB={km_db:.4f}  | CH={km_ch:.1f}")
print(f"   Hierarchical | K={best_k_ag:<2d} | Silhouette={hier_sil:.4f}  | DB={hier_db:.4f}  | CH={hier_ch:.1f}")
print(f"   DBSCAN       | K={n_clusters_db:<2d} | Silhouette={dbscan_sil:.4f}  | DB={dbscan_db:.4f}  | CH={dbscan_ch:.1f}")

print("\n📌 PART C — Anime Recommendation System (12,294 anime)")
print(f"   Content-Based: TF-IDF cosine similarity on genres")
print(f"   Collaborative SVD: Best k={best_k_svd}, RMSE={rmse_results[best_k_svd]:.4f}")
print("   Both recommenders fully functional ✅")
print("=" * 60)


: 